On s'occupe des tables des constructeurs

In [2]:
import numpy as np
import pandas as pd
#importation des données
cst = pd.read_csv("/Users/gabriels./Desktop/ENSAI /programmation /gab_projet_TDD/donnees_formule_un/constructors.csv")
cst_res = pd.read_csv("/Users/gabriels./Desktop/ENSAI /programmation /gab_projet_TDD/donnees_formule_un/constructor_results.csv")
cst_stand = pd.read_csv("/Users/gabriels./Desktop/ENSAI /programmation /gab_projet_TDD/donnees_formule_un/constructor_standings.csv")
race = pd.read_csv("/Users/gabriels./Desktop/projet_info2/donnees_formule_un/races.csv")
#cst_stand = pd.read_csv("donnees_formule_un/constructor_standings.csv")
#st_res = pd.read_csv("donnees_formule_un/constructor_results.csv")

structure de la table cst :
- constructorID = int
- constructorRef = object
- name = object
- nationality = object
- url = object

structure de la table cst_res:
- constructorResultsID = int
- raceID = int 
- constructorID = int
- points = float
- status = object

structure de la table cst_stand:
- constructorStandingsID = int
- raceID = int
- constructorID = int
- points = float
- positionText = object
- wins = int

In [64]:
#recherche des Na : on a des \\N = équivalent
cst
cst[cst.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

cst_stand
cst_stand[cst_stand.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

cst_res
cst_res[cst_res.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
# un \\N à chaque ligne


#on remplace les \N par des NA:
cst_res.replace("\\N", np.nan, inplace= True)


race.replace("\\N", np.nan, inplace= True)

Question : quelle écurie a gagné le plus de courses ? 
Faire un classement des écuries selon le nombre de victoires cumulées

In [168]:
#on fait un groubpy pour récupérer le constructorId ayant remporté le plus de points pour chaque course
max_course = cst_res.loc[:,['constructorId', 'raceId', 'points']].groupby('raceId').agg(premier = ('points', "max"))
#la colonne constructorId n'est pas affiché : on va merge avec d'autres tables pour récupérer les id puis les noms des constructeurs
max_course = pd.merge(max_course, race, on = ["raceId"])
max_course =  max_course[["raceId", "premier"]]
max_course
gagnants = pd.merge(
    max_course,
    cst_res,
    left_on=["raceId", "premier"],
    right_on=["raceId", "points"], #on fait correspondre "premier" et "points"
    how="inner"
)
gagnants = gagnants[["raceId", "constructorId", "points"]]
gagnants = pd.merge(gagnants, cst, on = ["constructorId"], how = "inner")
gagnants = gagnants[["raceId", "name", "points"]]
gagnants = gagnants.groupby("name", as_index= False).size()
gagnants.sort_values(by = "size", ascending= False)


,name,size
14,Ferrari,239
25,McLaren,192
42,Williams,126
27,Mercedes,117
31,Red Bull,111
36,Team Lotus,46
32,Renault,34
4,Benetton,28
5,Brabham,24
21,Lotus-Climax,22


La table ci-dessus indique que le contsructeur qui a cumulé le plus de courses remportées d'après la table race est ferrari avec 239 victoires 

Question 2: quel constructeur a remporté le plus de saisons ? 

In [171]:
#on veut récupérer la course la plus ancienne de la table race
race["date"].min()
#la première course de la base de donnée a été effectuée le 13/05/1950
season = pd.read_csv("/Users/gabriels./Desktop/projet_info2/donnees_formule_un/seasons.csv")
season #on remarque que les saisons sont découpées par années et qu'il n'y a pas de "débordement" d'une année à l'autre
#l'écurie qui aura remporté le plus de points dans la saison la remporte
#on veut regrouper à la fois par année et par constructeur

total_course = pd.merge(race, cst_res, how = "inner")
total_course = total_course.loc[:, ["raceId", "year", "constructorId", "points"]]
total_course

,raceId,year,constructorId,points
0,1,2009,23,18.0
1,1,2009,1,0.0
2,1,2009,7,11.0
3,1,2009,4,4.0
4,1,2009,3,3.0
...,...,...,...,...
12500,1132,2024,117,10.0
12501,1132,2024,3,2.0
12502,1132,2024,215,1.0
12503,1132,2024,15,0.0


In [203]:
total_course
race.groupby("year")["raceId"].count()
#on peut faire le groupby:

tbl = cst_res.loc[:,['constructorId', 'raceId', 'points']].groupby('raceId').agg(premier = ('points', "max"))
tbl = pd.merge(cst_res, tbl, left_on = ["raceId", "points"], right_on= ["raceId", "premier"])
tbl


,constructorResultsId,raceId,constructorId,points,status,premier
0,1,18,1,14.0,NaN,14.0
1,13,19,2,11.0,NaN,11.0
2,23,20,6,18.0,NaN,18.0
3,34,21,6,18.0,NaN,18.0
4,45,22,6,16.0,NaN,16.0
...,...,...,...,...,...,...
1087,16971,1129,1,28.0,NaN,28.0
1088,16972,1129,131,28.0,NaN,28.0
1089,16980,1130,9,29.0,NaN,29.0
1090,16990,1131,131,45.0,NaN,45.0


In [224]:

tbl2 = pd.merge(race, tbl, on = "raceId")
tbl2.columns
tbl2 = tbl2[["year", "raceId", "constructorId", "name", "points"]]
tbl2


,year,raceId,constructorId,name,points
0,2009,1,23,Australian Grand Prix,18.0
1,2009,2,23,Malaysian Grand Prix,7.0
2,2009,3,9,Chinese Grand Prix,18.0
3,2009,4,23,Bahrain Grand Prix,14.0
4,2009,5,23,Spanish Grand Prix,18.0
...,...,...,...,...,...
1087,2024,1129,1,Canadian Grand Prix,28.0
1088,2024,1129,131,Canadian Grand Prix,28.0
1089,2024,1130,9,Spanish Grand Prix,29.0
1090,2024,1131,131,Austrian Grand Prix,45.0


In [226]:
tbl2 = tbl2.groupby("year", as_index= False)["constructorId"].value_counts()
tbl2

,year,constructorId,count
0,1958,118,5
1,1958,6,3
2,1958,87,2
3,1959,170,5
4,1959,6,2
...,...,...,...
261,2023,131,1
262,2024,9,6
263,2024,1,3
264,2024,6,2


In [230]:
count_max = tbl2.groupby("year", as_index= False)["count"].max()
count_max
tbl2[tbl2["count"] == count_max["count"]]

ValueError: Can only compare identically-labeled Series objects